# LangChain Agents

## Tool Calling, ReAct Loop, Custom Tools, Tavily Search & Structured Responses

### What we will build

A simple **Research Agent** that can:

1. Answer normal questions using an LLM.
2. Use a calculator tool when a calculation is needed.
3. Search the web using Tavily when current information is needed.
4. Automatically choose the correct tool.
5. Return the final answer in a simple structured format.

### Learning flow

**LLM → Tool → Tool Calling → Agent → ReAct Loop → Custom Tools → Tavily → Structured Response**

> **Important:** We will learn the idea first and then write the code. Every important code section has comments explaining *what* we are doing and *why*.

## 1. What is an Agent?

A normal LLM application looks like:

```text
User → LLM → Answer
```

An **agent** can decide whether it needs to use a tool:

```text
User
  ↓
Agent / LLM
  ↓
Do I need a tool?
  ├── No  → Final Answer
  │
  └── Yes → Call Tool
              ↓
          Tool Result
              ↓
             LLM
              ↓
        Final Answer
```

### Simple analogy

- **LLM = Brain**
- **Tools = Hands**
- **Agent = Brain deciding when to use the hands**

We will start with a normal LLM and then gradually add these capabilities.

## 2. Install the Required Packages

Run this cell once.

### Packages used

- `langchain` → agent framework
- `langchain-google-genai` → connect LangChain to Gemini
- `langchain-tavily` → Tavily web-search tool
- `tavily-python` → Tavily dependency
- `python-dotenv` → load API keys from a `.env` file
- `pydantic` → define the structure of our final response

In [1]:
# Run this cell once if the packages are not installed.
# If you are using Google Colab/Jupyter, you can uncomment and run it.


In [4]:
%pip install -U langchain langchain-google-genai langchain-tavily tavily-python python-dotenv pydantic langchain-openrouter langchain-groq

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain-1.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_google_genai-4.5.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_tavily-0.2.18-py3-none-any.whl.metadata (20 kB)
  Using cached tavily_python-0.8.5-py3-none-any.whl.metadata (14 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached langchain_openrouter-0.2.9-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
  Using cached langgraph-1.2.22-py3-none-any.whl.metadata (4.9 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached langchain_protocol-0.0.21-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.17-py3-none-any.whl.metadata (39 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-win_amd64.whl.metadata (6.5 kB

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


## 3. API Keys

For this notebook we use:

- **Google Gemini** as the LLM
- **Tavily** for web search

Create a `.env` file in the same folder as this notebook:

```text
GOOGLE_API_KEY=your_google_api_key
TAVILY_API_KEY=your_tavily_api_key
```

### Why do we need API keys?

The Python code needs permission to communicate with these external services.

> Never hard-code real API keys directly into a notebook that you plan to share publicly.

In [2]:
# Load the API keys from the .env file

from dotenv import load_dotenv

load_dotenv()

print("API keys loaded from .env")

API keys loaded from .env


## 4. Import the Gemini Chat Model

First, let's use Gemini just like a normal LLM.

At this stage there is **no agent and no tool**.

In [11]:



from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openai/gpt-oss-20b",
    temperature=0,
)
print("OpenRouter model created successfully.")

OpenRouter model created successfully.


## 5. Test the LLM

Let's ask a simple question.

This is the basic pattern:

```text
Question → LLM → Answer
```

In [12]:
# Ask the model a simple question

response = model.invoke("What is the current temperature of bengaluru ?")

# .content contains the text generated by the model
print(response.content)

I don’t have access to live data, so I can’t give you the exact temperature right now. Bengaluru (Bangalore) usually has a mild, tropical‑savanna climate with temperatures that hover between **20 °C (68 °F) and 30 °C (86 °F)** for most of the year.  

If you need the current temperature, you can check one of the following:

| Source | How to use |
|--------|------------|
| **Weather websites** (e.g., Weather.com, AccuWeather, BBC Weather) | Search “Bengaluru” or “Bangalore” and view the current reading. |
| **Weather apps** (e.g., The Weather Channel, Weather Underground, Google Weather) | Open the app and look up Bengaluru. |
| **Search engines** | Type “Bengaluru weather” or “current temperature in Bengaluru” into Google, Bing, etc. |
| **Smart assistants** | Ask Siri, Google Assistant, or Alexa: “What’s the temperature in Bengaluru?” |

These sources pull real‑time data from local weather stations and satellite feeds, so they’ll give you the most accurate, up‑to‑minute reading.


In [13]:
# Ask the model a simple question

response = model.invoke("What is the current cm of tamil nadu ?")

# .content contains the text generated by the model
print(response.content)

The current Chief Minister of Tamil Nadu is **M. K. Stalin**. He has been in office since 7 May 2021, following the victory of the Dravida Munnetra Kazhagam (DMK) in the 2021 state elections.


# Part A — Create Our First Tool

## 6. What is a Tool?

A **tool is a Python function that an agent is allowed to use**.

For example, we can create a function that multiplies two numbers.

The important difference is:

```text
Normal Python function
        ↓
Python program decides when to call it
```

With an agent:

```text
User question
      ↓
LLM decides whether the tool is useful
      ↓
Tool is called
```

In [14]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""

    # This is the actual Python work done by the tool.
    return a * b

## 7. Why did we use `@tool`?

The `@tool` decorator tells LangChain:

> "Make this Python function available as an LLM tool."

### Why is the docstring important?

```python
"""Multiply two numbers together."""
```

The LLM uses the tool's name, description, and input information to understand **when the tool should be used**.

Good tool descriptions help the agent select the correct tool.

In [15]:
# We can still call the tool directly.
# This is useful for understanding that a tool is still a Python function.

result = multiply.invoke({
    "a": 10,
    "b": 20
})

print("Result:", result)

Result: 200


Expected output:

```text
Result: 200
```

Notice:

**We called the tool ourselves.**

Next, we will let the **agent decide when to call it**.

# Part B — Build Our First Agent

## 8. Create an Agent with `create_agent`

LangChain provides `create_agent()` to create an agent.

We give it two things:

1. `model` → the LLM that makes decisions
2. `tools` → the tools the agent is allowed to use

So:

```text
Gemini Model + Tools
          ↓
        Agent
```

In [16]:
from langchain.agents import create_agent

# Create an agent and give it our multiply tool.
agent = create_agent(
    model=model,
    tools=[multiply]
)

print("Agent created successfully.")

Agent created successfully.


## 9. Ask the Agent to Calculate

Now we don't call `multiply()` ourselves.

Instead, we ask the **agent** a question.

The agent can decide:

> "This is a multiplication question, so I should use the multiply tool."

In [17]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 25 multiplied by 12?"
        }
    ]
})

# The agent keeps its conversation in the "messages" state.


# The last message normally contains the final response.
print(result["messages"][-1].content)

25 multiplied by 12 equals **300**.


## 10. What Happened Behind the Scenes?

Conceptually, the agent performed:

```text
User
 ↓
"What is 25 × 12?"
 ↓
LLM
 ↓
"I should use the multiply tool"
 ↓
multiply(25, 12)
 ↓
300
 ↓
LLM
 ↓
Final Answer
```

This is **tool calling**.

The LLM does not directly execute Python code.

Instead, it requests a tool call, and the agent runtime executes the tool.

# Part C — Understand the ReAct Loop

## 11. What is ReAct?

ReAct means:

**Reason + Act**

The useful model is:

```text
Reason → Act → Observe → Repeat
```

For example:

```text
User asks a question
       ↓
Agent decides what to do
       ↓
Act: call a tool
       ↓
Observe: receive tool result
       ↓
Decide what to do next
       ↓
Final Answer
```

We are using "reason" here as a **conceptual description of the decision process**. We should not expose private chain-of-thought to users.

## 12. ReAct Example

Suppose the user asks:

> "Add 20 and 30, then multiply the result by 5."

The agent can perform:

```text
User
 ↓
Add 20 + 30
 ↓
Observe: 50
 ↓
Multiply 50 × 5
 ↓
Observe: 250
 ↓
Final Answer: 250
```

This demonstrates why agents are useful: the next action can depend on the result of the previous action.

# Part D — Add Another Custom Tool

## 13. Create an Addition Tool

Let's give our agent another capability.

Now it can choose between:

- `add`
- `multiply`

In [18]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""

    # Perform the addition and return the result.
    return a + b

In [19]:
# Give both tools to the agent.
agent = create_agent(
    model=model,
    tools=[add, multiply]
)

print("Agent now has 2 tools.")

Agent now has 2 tools.


## 14. Test the Addition Tool

Ask:

> What is 100 + 250?

The important part is that **we don't tell the agent which tool to use**.

The agent should select `add`.

In [20]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 100 + 250?"
        }
    ]
})

print(result)
#print(result["messages"][-1].content)

{'messages': [HumanMessage(content='What is 100 + 250?', additional_kwargs={}, response_metadata={}, id='b1c64a16-62ba-4fcb-8205-a057ce37a114'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the add function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to use the add function.'}]}, response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1794760916-mmvTFPPXG3w6NJQK6C8C', 'created': 1794760916, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 5.166e-06, 'cost_details': {'upstream_inference_completions_cost': 3.15e-06, 'upstream_inference_prompt_cost': 2.016e-06, 'upstream_inference_cost': 5.166e-06}}, id='lc_run--01a1e016-3b80-716d-b46c-54d5624a5564-0', tool_calls=[{'name': 'add', 'args': {'a': 100, 'b': 250}, 'id': 'chatcmpl-tool-918c53406f0af7eb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

## 15. Test Multiple Tool Calls

Now try a question that requires two steps:

> Add 20 and 30, then multiply the result by 5.

The agent may follow this conceptual process:

```text
add(20, 30)
      ↓
     50
      ↓
multiply(50, 5)
      ↓
    250
```

This is our first simple multi-step agent example.

In [21]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add 20 and 30, then multiply the result by 5."
        }
    ]
})

print(result["messages"][-1].content)

The result is **250**.


## 16. Inspect the Agent Messages

For learning purposes, let's look at the messages returned by the agent.

This helps us understand that an agent interaction can contain:

```text
User message
     ↓
AI tool call
     ↓
Tool result
     ↓
AI final answer
```

In [22]:
# Print each message so we can see the agent's execution state.

for i, message in enumerate(result["messages"], start=1):
    print(f"--- Message {i} ---")
    print("Type:", type(message).__name__)
    print(message)
    print()

--- Message 1 ---
Type: HumanMessage
content='Add 20 and 30, then multiply the result by 5.' additional_kwargs={} response_metadata={} id='476f8fd4-f3f0-40c0-b168-c757a41b6979'

--- Message 2 ---
Type: AIMessage
content='' additional_kwargs={'reasoning_content': "We need to use the add function to add 20 and 30, then multiply the result by 5. We'll call add first, then multiply.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "We need to use the add function to add 20 and 30, then multiply the result by 5. We'll call add first, then multiply."}]} response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1794760925-NQVUEc0bDoNlQTgj39Ej', 'created': 1794760925, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 6.93e-06, 'cost_details': {'upstream_inference_completions_cost': 5.22e-06, 'upstream_inference_prompt_cost': 1.71e-06, 'upstream_inference_cost': 6.93e-06}} id='

### Looking for tool calls

AI messages can contain a `tool_calls` attribute.

Let's inspect it when available.

In [23]:
for message in result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:")
        print(message.tool_calls)
        print()

Tool calls:
[{'name': 'add', 'args': {'a': 20, 'b': 30}, 'id': 'chatcmpl-tool-a3d736ee80dfdea3', 'type': 'tool_call'}]

Tool calls:
[{'name': 'multiply', 'args': {'a': 50, 'b': 5}, 'id': 'chatcmpl-tool-900e92c21405d617', 'type': 'tool_call'}]



# Part E — Add Tavily Web Search

## 17. Why Do We Need Web Search?

Ask the LLM:

> "What is the latest news about AI?"

An LLM may not have live web access in the way a search engine does.

So we give the agent a **web-search tool**.

The architecture becomes:

```text
User
 ↓
Agent
 ↓
LLM
 ↓
Tavily Search
 ↓
Search Results
 ↓
LLM
 ↓
Final Answer
```

In [24]:
from langchain_tavily import TavilySearch

# Create a Tavily search tool.

# max_results controls how many search results are returned.
search = TavilySearch(
    max_results=5
)

print("Tavily search tool created.")

Tavily search tool created.


## 18. Give Tavily to the Agent

Now our agent has three tools:

```text
1. add
2. multiply
3. web search
```

The agent can decide which one is appropriate.

In [25]:
tools = [
    add,
    multiply,
    search
]

agent = create_agent(
    model=model,
    tools=tools
)

print("Agent created with 3 tools.")

Agent created with 3 tools.


## 19. Test Web Search

Ask a question that clearly needs current information.

For example:

> "Search the web for the latest information about AI agents."

The agent should recognize that web search is useful.

In [26]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Search the web for the latest information about AI agents."
        }
    ]
})

print(result["messages"][-1].content)

**Latest developments in AI agents (up to March 2026)**  

| Source | Key take‑aways | Date / Relevance |
|--------|----------------|------------------|
| **Wikipedia – “AI agent”** | • 2018‑2023: academic research on LLM agents. <br>• 2023‑2024: rapid deployment after OpenAI’s function‑calling API and Anthropic’s Model Context Protocol (MCP). <br>• 2024‑2025: agents used in gaming, gambling, crypto, social media, and software development. <br>• 2025: companies report mixed ROI; government contracts (e.g., U.S. DoD) begin to use agents for operational decision‑making. | 2025‑2026 (latest revision) |
| **Mastra.ai – “AI Agents News: Latest Developments and Trends”** | • Agent adoption is now common across small, mid‑size, and enterprise firms. <br>• MCP is becoming a neutral, vendor‑agnostic standard (donated to the Linux Foundation in Dec 2025). <br>• Memory & context engineering is critical for coherent, multi‑turn agents. | Published 2025 |
| **MIT AI Agent Index (2025 Edition)** | •

In [27]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "who is the cm of tamil nadu ?"
        }
    ]
})

print(result["messages"][-1].content)

The current Chief Minister of Tamil Nadu is **M. K. Stalin** (Dravida Munnetra Kazhagam). He was sworn in on 7 May 2021 and has been serving in that role since then.  

*Source: Wikipedia – “M. K. Stalin” (Chief Minister of Tamil Nadu, 2021‑present)*


# Part F — The Main Agent Demonstration

## 20. Combine Search + Calculation

Now we will ask the agent a question that requires **more than one tool**.

Example:

> "Search the web for India's current population and calculate 2% of it."

The agent needs to:

1. Search for current population information.
2. Read the search result.
3. Perform a calculation.
4. Give the final answer.

This is a very good example of the **ReAct-style loop**.

In [28]:
question = (
    "Search the web for India's current population "
    "and calculate 2% of that population."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

print(result["messages"][-1].content)

**India’s current population (mid‑2024 estimate)**  
- **1,450,935,791** people (Worldometer, 2024 mid‑year estimate)【3†L1-L4】  

**2 % of that population**  
- 0.02 × 1,450,935,791 ≈ **29,018,716** people.  

So, about **29 million** people represent 2 % of India’s current population.


## 21. Understand This Execution

Conceptually:

```text
User
 ↓
Agent
 ↓
LLM
 ↓
Need current population
 ↓
Tavily Search
 ↓
Population information
 ↓
LLM
 ↓
Need calculation
 ↓
Calculator Tool
 ↓
Result
 ↓
LLM
 ↓
Final Answer
```

### The key idea

The agent chooses the **next action based on the current situation**.

That is what makes it different from a fixed chain.

# Part G — Tool Calling vs Agent

## 22. Important Difference

### Tool Calling

The LLM can request:

```text
"Please call the multiply tool with a=10 and b=20."
```

### Agent

The system can repeatedly decide:

```text
Should I answer?
Should I search?
Should I calculate?
Should I call another tool?
Am I finished?
```

So remember:

> **Tool calling is a capability. An agent is a system that uses tool calling to make decisions and complete tasks.**

# Part H — Response Formatting

## 23. Why Structured Responses?

So far, the agent returns normal text.

For example:

```text
The answer is 250.
```

But applications often need predictable data.

For example:

```json
{
    "answer": "250",
    "summary": "The calculation was completed.",
    "sources": []
}
```

This is easier for a Python program, API, or frontend to consume.

In [29]:
from pydantic import BaseModel, Field

class AgentResponse(BaseModel):
    # The main answer shown to the user.
    answer: str = Field(
        description="The final answer to the user's question."
    )

    # A short explanation/summary.
    summary: str = Field(
        description="A short summary of the result."
    )

    # URLs or source names used during research.
    sources: list[str] = Field(
        description="Important sources used to produce the answer."
    )

print("Response schema created.")

Response schema created.


## 24. Create an Agent with Structured Response

We can tell `create_agent()` that we want the final response to follow our Pydantic schema.

The important idea is:

```text
Agent
 ↓
Final result
 ↓
Follow AgentResponse schema
```

This is different from tool calling.

- **Tool calling** → the model requests an action.
- **Structured response** → the final answer follows a predictable format.

In [30]:
agent_structured = create_agent(
    model=model,
    tools=tools,
    response_format=AgentResponse
)

print("Structured-output agent created.")

Structured-output agent created.


## 25. Test Structured Output

Let's ask a simple research question.

In [31]:
result = agent_structured.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Search the web and explain what an AI agent is in simple words."
        }
    ]
})

# create_agent stores the structured final result in "structured_response".
response = result["structured_response"]

print("ANSWER:")
print(response.answer)

print("\nSUMMARY:")
print(response.summary)

print("\nSOURCES:")
for source in response.sources:
    print("-", source)

ANSWER:
An **AI agent** is a piece of software that can *think* and *act* on its own to reach a goal you give it. Think of it like a helpful robot that follows a set of instructions, but instead of just repeating the same steps, it can decide which step to take next, use tools (like a calculator, a web‑browser, or a database), and learn from what it does. In short: it takes a goal, plans the steps, and carries them out—adjusting its plan as it goes.

Sources:
- AWS: “An AI agent is a software program that can interact with its environment, collect data, and use that data to perform self‑directed tasks that meet predetermined goals.”
- Google Cloud: “AI agents are software systems that use AI to pursue goals and complete tasks on behalf of users. They show reasoning, planning, and memory and have a level of autonomy to make decisions, learn, and adapt.”

These descriptions capture the core idea: an AI agent is a goal‑driven, self‑directed system that can plan, act, and adapt to achieve 

# Part I — Final Mini Project

## 26. Build a Research & Calculator Agent

Our final agent has:

### Model

```text
Gemini
```

### Tools

```text
Tavily Search
Add
Multiply
```

### Response format

```text
AgentResponse
 ├── answer
 ├── summary
 └── sources
```

### Architecture

```text
                    USER
                      ↓
                    AGENT
                      ↓
                     LLM
                      ↓
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       Search        Add       Multiply
          │           │           │
          └───────────┼───────────┘
                      ↓
                 Tool Result
                      ↓
                     LLM
                      ↓
              Structured Response
                      ↓
                    USER
```

In [45]:
import os
from typing import List
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_openrouter import ChatOpenRouter
from langchain_core.tools import tool

load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY not found"

model = ChatOpenRouter(
    model="openai/gpt-oss-20b",
    temperature=0,
)

# Every field you print later MUST be defined here
class AgentResponse(BaseModel):
    answer: str = Field(description="The final answer to the user's question")
    summary: str = Field(description="A one or two sentence summary of the answer")
    sources: List[str] = Field(
        default_factory=list,
        description="Sources, tools or references used. Empty list if none.",
    )

# Keep your own tools here; this one is a placeholder
@tool
def get_word_length(word: str) -> int:
    """Return the number of characters in a word."""
    return len(word)

agent = create_agent(
    model=model,
    tools=[get_word_length],
    response_format=ToolStrategy(AgentResponse),
)

In [46]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "How many letters are in the word 'unbelievable'?"}]}
)

response = result["structured_response"]

print("ANSWER")
print(response.answer)

print("\nSUMMARY")
print(response.summary)

print("\nSOURCES")
if response.sources:
    for source in response.sources:
        print("-", source)
else:
    print("No sources returned.")

ANSWER
The word 'unbelievable' has 12 letters.

SUMMARY
'unbelievable' contains 12 letters.

SOURCES
No sources returned.


In [47]:
print(type(result["structured_response"]))
print(result["structured_response"])

<class '__main__.AgentResponse'>
answer="The word 'unbelievable' has 12 letters." summary="'unbelievable' contains 12 letters." sources=[]


# Part J — Student Practice Exercises

## Challenge 1 — Calculator

Ask the agent:

```text
What is 125 multiplied by 48?
```

**Expected behavior:** use the `multiply` tool.

---

## Challenge 2 — Addition

Ask:

```text
What is 450 + 275?
```

**Expected behavior:** use the `add` tool.

---

## Challenge 3 — Multiple Tools

Ask:

```text
Add 100 and 200, then multiply the result by 5.
```

Expected conceptual flow:

```text
add
 ↓
300
 ↓
multiply
 ↓
1500
```

---

## Challenge 4 — Web Search

Ask:

```text
Search the web for the latest developments in generative AI.
```

**Expected behavior:** use Tavily.

---

## Challenge 5 — Search + Calculation

Ask:

```text
Search the web for India's current population
and calculate 5% of it.
```

The agent should decide that it needs:

```text
Tavily → population information
       ↓
Calculation → 5%
```

---

# Final Challenge

Build your own tool.

For example:

```python
@tool
def square(number: int) -> int:
    """Return the square of a number."""
    return number * number
```

Add it to the agent and test:

```text
What is the square of 25?
```

# 27. Final Summary

You have built an agent step by step.

## Step 1 — Normal LLM

```text
User → LLM → Answer
```

## Step 2 — Create a Tool

```python
@tool
def multiply(...):
    ...
```

## Step 3 — Create an Agent

```python
agent = create_agent(
    model=model,
    tools=[multiply]
)
```

## Step 4 — Add More Tools

```text
add
multiply
```

## Step 5 — Add Tavily

```text
web search
```

## Step 6 — Understand ReAct

```text
Reason → Act → Observe → Repeat
```

## Step 7 — Structured Response

```python
class AgentResponse(BaseModel):
    answer: str
    summary: str
    sources: list[str]
```

---

# The Mental Model to Remember

```text
LLM
 +
Tools
 +
Decision Making
 +
Loop
 =
Agent
```

And:

```text
Tool Calling → "Which tool should I call?"

Agent → "What should I do next to solve the user's task?"

ReAct → "Reason/decide → Act → Observe → Repeat"

Structured Output → "Give the final result in a predictable format."
```

# 28. Troubleshooting

## API key error

Check that `.env` contains:

```text
GOOGLE_API_KEY=...
TAVILY_API_KEY=...
```

Then restart the notebook kernel if necessary and run:

```python
from dotenv import load_dotenv
load_dotenv()
```

---

## Tavily error

Check that:

```python
search = TavilySearch(max_results=5)
```

was created successfully and that `TAVILY_API_KEY` is available.

---

## Import error

Make sure the packages are installed in the **same Python environment/kernel** that is running this notebook.

You can check the Python executable with:

```python
import sys
print(sys.executable)
```

Then rerun the installation cell in that environment.

---

## Important version note

LangChain's APIs evolve quickly. If an import or argument differs in a future release, check the installed package version before changing the code.

You can see the installed version with:

```python
import langchain
print(langchain.__version__)
```